Amazon Electronics Reviews

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

Download Raw Structured Data

In [ ]:
from src.data.download import download_electronics_reviews
# keep default force_download as False to avoid unnecessary downloads
download_electronics_reviews(force_download=False)

100%|██████████| 109M/109M [00:21<00:00, 5.28MB/s] 

Extracting files...


Dataset downloaded to C:\Users\harsh\IISCCapstoneProjectGroup16\data\raw
Dataset renamed to C:\Users\harsh\IISCCapstoneProjectGroup16\data\raw\ratings_electronics.csv


Load Raw Structured Data

In [4]:
from src.data.loaders import load_electronics_reviews
df = load_electronics_reviews()
print(df.head())

          user_id  product_id  rating   timestamp
0   AKM1MP6P0OYPR  0132793040     5.0  1365811200
1  A2CX7LUOHB2NDG  0321732944     5.0  1341100800
2  A2NWSAGRHCP8N5  0439886341     1.0  1367193600
3  A2WNBOD3WNDNKT  0439886341     3.0  1374451200
4  A1GI0U4ZRJA8WN  0439886341     1.0  1334707200


Process Structured Data

In [5]:
from src.data.preprocessing import preprocess_ratings, save_processed_data
df_preprocessed = preprocess_ratings(df)
save_processed_data(df_preprocessed)
print(df_preprocessed.head())

Initial shape: (7824482, 4)
Final shape: (7824482, 4)
Saved to: C:\Users\harsh\IISCCapstoneProjectGroup16\data\processed\ratings_electronics_processed.csv
          user_id  product_id  rating  timestamp
0   AKM1MP6P0OYPR  0132793040     5.0 2013-04-13
1  A2CX7LUOHB2NDG  0321732944     5.0 2012-07-01
2  A2NWSAGRHCP8N5  0439886341     1.0 2013-04-29
3  A2WNBOD3WNDNKT  0439886341     3.0 2013-07-22
4  A1GI0U4ZRJA8WN  0439886341     1.0 2012-04-18


Upload Structured data to PostgreSQL DB

In [ ]:
from src.data.postgresql import upload_dataframe_to_postgresql_db
# options for if_exists: 'fail', 'replace', or 'append'
# keep default as "fail" to not overwrite existing data (unless there is a need to "replace" it)
upload_dataframe_to_postgresql_db(df_preprocessed, if_exists="replace")

482

Execute Sample Query

In [ ]:
from src.data.postgresql import execute_sql_query
from src.config.data import POSTGRESQL_TABLE_NAME

query_sql = "SELECT VERSION();"
print(execute_sql_query(query_sql))

query_sql = f"SELECT COUNT(*) FROM {POSTGRESQL_TABLE_NAME};"
print(execute_sql_query(query_sql))

7824482


In [12]:
# Function that executes a SQL query on the PostgreSQL database and returns the result set
import psycopg2

def execute_sql_query(query: str):
    try:
        # Connect to the PostgreSQL database
        conn = psycopg2.connect(POSTGRES_URI)
        # Create a cursor to perform database operations
        cur = conn.cursor()
        # Execute the query
        cur.execute(query)
        # Fetch all results
        columns = [desc[0] for desc in cur.description]
        results = cur.fetchall()
        # Close communication with the database
        cur.close()
        conn.close()
        # Return as a list of dictionaries for easier handling by the LLM
        return [dict(zip(columns, row)) for row in results]
    except Exception as e:
        return f"Error executing query: {e}"

In [13]:
execute_sql_query("SELECT * FROM ratings_electronics LIMIT 5;")

'Error executing query: connection to server at "pg-20bad560-myproject123-456.e.aivencloud.com" (168.144.36.107), port 12548 failed: Connection refused (0x0000274D/10061)\n\tIs the server running on that host and accepting TCP/IP connections?\n'

Generate Synthetic Reviews

In [ ]:
from src.data.synthetic_generator import generate_synthetic_queries, save_synthetic_queries
df = generate_synthetic_queries(total_queries=200)
save_synthetic_queries(df)

PosixPath('/Users/shurtugal/Desktop/IISC/CAPSTONE PROJECT/IISCCapstoneProjectGroup16/data/synthetic/synthetic_queries_20260610_192417.csv')